###  Notebook 3: Demand Prediction Agent

**Purpose:** Predict per-station utilization rate and congestion probability at each 5-minute interval using historical session features.

**Models Trained:**
- **Random Forest Regressor** — baseline model
- **LightGBM Regressor** — main utilization prediction model
- **LightGBM Classifier** — congestion probability model

**Evaluation Metrics:**
- RMSE — penalizes large prediction errors
- MAE — average absolute error
- R² — variance explained by the model

**Assumption:** Lagged utilization (t-1 to t-48) is the strongest signal. Time-based train/test split is used (no shuffle) to preserve temporal order.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                              r2_score, classification_report, accuracy_score)
import lightgbm as lgb
import joblib
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 130

BASE  = os.path.normpath(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
PROC  = os.path.join(BASE, "data", "processed") + os.sep
PLOTS = os.path.join(BASE, "plots")             + os.sep
OUT   = os.path.join(BASE, "outputs")           + os.sep

print("✅ Libraries loaded")
print(f"PROC  : {PROC}")
print(f"OUT   : {OUT}")

In [ ]:
panel = pd.read_csv(PROC + "urbanev_panel.csv", parse_dates=['datetime'])
panel = panel.sort_values(['station_id','datetime']).reset_index(drop=True)
print(f"Panel shape: {panel.shape}")
panel[['station_id','datetime','utilization','price','hour','day_of_week','congestion']].head(3)

### 3.1 Feature Engineering — Lag & Rolling Features
Lag features capture temporal autocorrelation in charging demand. Rolling statistics capture short-run trends.

In [ ]:
panel = panel.sort_values(['station_id','datetime'])
grp   = panel.groupby('station_id')['utilization']

# Lagged utilization features
for lag in [1, 3, 6, 12, 24, 48]:
    panel[f'util_lag_{lag}'] = grp.shift(lag)

# Rolling statistics
panel['util_roll_mean_12'] = grp.transform(lambda x: x.shift(1).rolling(12).mean())
panel['util_roll_std_12']  = grp.transform(lambda x: x.shift(1).rolling(12).std())
panel['util_roll_mean_48'] = grp.transform(lambda x: x.shift(1).rolling(48).mean())

# Same hour yesterday (288 steps = 24hrs × 12 intervals/hr)
panel['util_lag_288']  = grp.shift(288)
# Same hour last week
panel['util_lag_2016'] = grp.shift(2016)

# Drop rows where critical lags are NaN
panel_model = panel.dropna(subset=[f'util_lag_{l}' for l in [1, 3, 6, 12]]).copy()

print(f"✅ After lag features: {panel_model.shape}")
print(f"   Features added: util_lag_1/3/6/12/24/48/288/2016, roll_mean_12/48, roll_std_12")

In [ ]:
FEATURES = [
    'hour', 'day_of_week', 'is_weekend', 'month',
    'price', 'volume', 'duration',
    'util_lag_1', 'util_lag_3', 'util_lag_6', 'util_lag_12',
    'util_lag_24', 'util_lag_48', 'util_lag_288',
    'util_roll_mean_12', 'util_roll_std_12', 'util_roll_mean_48',
    'CBD', 'fast_count', 'slow_count', 'count', 'area'
]
FEATURES = [f for f in FEATURES if f in panel_model.columns]

TARGET_REG = 'utilization'
TARGET_CLS = 'congestion'

X     = panel_model[FEATURES].fillna(0)
y_reg = panel_model[TARGET_REG]
y_cls = panel_model[TARGET_CLS]

# Time-based split — NEVER shuffle a time-series
split_idx = int(len(X) * 0.8)
X_train, X_test         = X.iloc[:split_idx],     X.iloc[split_idx:]
y_reg_train, y_reg_test = y_reg.iloc[:split_idx], y_reg.iloc[split_idx:]
y_cls_train, y_cls_test = y_cls.iloc[:split_idx], y_cls.iloc[split_idx:]

print(f"Train : {X_train.shape} | Test : {X_test.shape}")
print(f"Features used : {len(FEATURES)}")
print(f"Congestion rate — Train: {y_cls_train.mean():.2%} | Test: {y_cls_test.mean():.2%}")

### 3.2 Model A — Random Forest Regressor (Baseline)

In [ ]:
rf = RandomForestRegressor(
    n_estimators=100, max_depth=10,
    n_jobs=-1, random_state=42
)
rf.fit(X_train, y_reg_train)
rf_pred = rf.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_reg_test, rf_pred))
rf_mae  = mean_absolute_error(y_reg_test, rf_pred)
rf_r2   = r2_score(y_reg_test, rf_pred)

print("=" * 45)
print("  Random Forest — Utilization Prediction")
print("=" * 45)
print(f"  RMSE : {rf_rmse:.4f}")
print(f"  MAE  : {rf_mae:.4f}")
print(f"  R²   : {rf_r2:.4f}")

### 3.3 Model B — LightGBM Regressor (Main Model)

In [ ]:
lgb_reg = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05,
    max_depth=8, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1
)
lgb_reg.fit(
    X_train, y_reg_train,
    eval_set=[(X_test, y_reg_test)],
    callbacks=[lgb.early_stopping(50, verbose=False),
               lgb.log_evaluation(100)]
)

lgb_pred = lgb_reg.predict(X_test)
lgb_rmse = np.sqrt(mean_squared_error(y_reg_test, lgb_pred))
lgb_mae  = mean_absolute_error(y_reg_test, lgb_pred)
lgb_r2   = r2_score(y_reg_test, lgb_pred)

print("=" * 45)
print("  LightGBM — Utilization Prediction")
print("=" * 45)
print(f"  RMSE : {lgb_rmse:.4f}")
print(f"  MAE  : {lgb_mae:.4f}")
print(f"  R²   : {lgb_r2:.4f}")

### 3.4 Model C — LightGBM Classifier (Congestion Probability)

In [ ]:
scale_weight = (y_cls_train == 0).sum() / (y_cls_train == 1).sum()

lgb_cls = lgb.LGBMClassifier(
    n_estimators=300, learning_rate=0.05,
    max_depth=6, num_leaves=31,
    scale_pos_weight=scale_weight,
    random_state=42, n_jobs=-1
)
lgb_cls.fit(
    X_train, y_cls_train,
    eval_set=[(X_test, y_cls_test)],
    callbacks=[lgb.early_stopping(50, verbose=False),
               lgb.log_evaluation(100)]
)

cls_pred = lgb_cls.predict(X_test)
cls_prob = lgb_cls.predict_proba(X_test)[:, 1]

print("=" * 50)
print("  LightGBM — Congestion Classification Report")
print("=" * 50)
print(classification_report(y_cls_test, cls_pred,
                             target_names=['Normal','Congested']))
print(f"  Accuracy : {accuracy_score(y_cls_test, cls_pred):.4f}")

### 3.5 Metrics Comparison & Visualizations

In [ ]:
metrics_df = pd.DataFrame({
    'Model': ['Random Forest (Baseline)', 'LightGBM (Main)'],
    'RMSE' : [rf_rmse, lgb_rmse],
    'MAE'  : [rf_mae,  lgb_mae],
    'R²'   : [rf_r2,   lgb_r2]
})
metrics_df.to_csv(OUT + "demand_prediction_metrics.csv", index=False)
print(metrics_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Actual vs Predicted scatter
idx = np.random.choice(len(y_reg_test), min(1000, len(y_reg_test)), replace=False)
axes[0].scatter(y_reg_test.iloc[idx], lgb_pred[idx],
                alpha=0.2, s=8, color='steelblue')
axes[0].plot([0,1],[0,1],'r--', lw=1.5)
axes[0].set_xlabel('Actual Utilization')
axes[0].set_ylabel('Predicted Utilization')
axes[0].set_title(f'LightGBM: Actual vs Predicted\nR²={lgb_r2:.3f}  RMSE={lgb_rmse:.4f}',
                   fontsize=11, fontweight='bold')

# RMSE comparison bar
axes[1].bar(['Random Forest','LightGBM'], [rf_rmse, lgb_rmse],
            color=['#3498db','#e74c3c'], edgecolor='white')
axes[1].set_title('RMSE Comparison', fontsize=11, fontweight='bold')
axes[1].set_ylabel('RMSE')
for i, v in enumerate([rf_rmse, lgb_rmse]):
    axes[1].text(i, v + 0.001, f"{v:.4f}", ha='center', fontweight='bold')

# Feature importance
fi = pd.DataFrame({
    'feature'   : FEATURES,
    'importance': lgb_reg.feature_importances_
}).sort_values('importance', ascending=True).tail(12)
fi.plot(kind='barh', x='feature', y='importance',
        ax=axes[2], color='steelblue', legend=False)
axes[2].set_title('Top 12 Feature Importances (LightGBM)', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Importance')

plt.tight_layout()
plt.savefig(PLOTS + "demand_prediction_results.png", bbox_inches='tight')
plt.show()
print("📁 Saved: demand_prediction_results.png")

In [ ]:
# Time-series: Actual vs Predicted for one sample station
one_st_id  = panel_model['station_id'].unique()[0]
one_station = panel_model[panel_model['station_id'] == one_st_id].copy()

if len(one_station) > 0:
    X_st     = one_station[FEATURES].fillna(0)
    preds_st = lgb_reg.predict(X_st)
    show_n   = min(288, len(one_station))  # 24 hrs of 5-min data

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(one_station['datetime'].values[:show_n],
            one_station['utilization'].values[:show_n],
            label='Actual', color='steelblue', lw=1.5)
    ax.plot(one_station['datetime'].values[:show_n],
            preds_st[:show_n],
            label='Predicted', color='coral', lw=1.5, ls='--')
    ax.set_title(f'Demand Forecast: Actual vs Predicted — Station {one_st_id} (24 hrs sample)',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Utilization Rate')
    ax.set_xlabel('Datetime')
    ax.legend()
    plt.tight_layout()
    plt.savefig(PLOTS + "demand_timeseries_forecast.png", bbox_inches='tight')
    plt.show()
    print("📁 Saved: demand_timeseries_forecast.png")

In [ ]:
# Save models
joblib.dump(lgb_reg, OUT + "lgb_demand_regressor.pkl")
joblib.dump(lgb_cls, OUT + "lgb_congestion_classifier.pkl")

# Save test predictions for use by Notebook 4
test_preds = panel_model.iloc[split_idx:][
    ['datetime','station_id','utilization','congestion']].copy()
test_preds['pred_utilization']      = lgb_pred
test_preds['pred_congestion_prob']  = cls_prob
test_preds['pred_congestion']       = cls_pred
test_preds.to_csv(OUT + "demand_predictions.csv", index=False)

print("✅ Models saved:")
print(f"   lgb_demand_regressor.pkl")
print(f"   lgb_congestion_classifier.pkl")
print(f"   demand_predictions.csv → {test_preds.shape[0]:,} rows")
print("\n✅ Notebook 3 complete! Run Notebook 4 next.")